# Track 3 — Intraday seasonality: calendar-adjusted volume surprises (Priyal)

**Hypothesis (written before the full run):** The incumbent 15:30 ridge sees `cp_volume_surprise20` (volume by 15:30 ÷ its 20-day average) and calendar flags, but only as separate additive terms. On predictable heavy days (month-end, quarterly expiry, Russell/MSCI/S&P rebalances, FOMC), two things go wrong:

1. The volume by 15:30 looks like a surprise when it is normal for that kind of day, and past special days inflate the 20-day baseline.
2. The share of the day's volume that goes into the closing auction jumps on those days, and the size of the jump scales with each stock's own checkpoint volume. A single additive flag can't capture that.

So I expect calendar-adjusted features to cut the correction error mainly on **month-end, quarterly-expiry and rebalance days**, with little change on ordinary days. I also expect the index-rebalance events (from `blackrock_data/rebalancing/`) to account for a meaningful part of the gain.

**Features** (all in `track3_features.py`, refitted for each test year on earlier rows only):

| Feature | Meaning |
|---|---|
| `cal_surprise` | log(volume by 15:30) − expected day effect − log(clean 20-day baseline): how unusual today is *for this kind of day* |
| `cal_expected_cp` | expected day effect on 15:30 volume, from a regression on day-type dummies |
| `cal_expected_ratio` | expected day effect on the auction ÷ 15:30 volume ratio |
| `cal_rule_correction` | log1p(15:30 volume × clean ratio baseline × expected ratio uplift) − log daily forecast: a calendar-aware rule forecast, written as a correction |
| `cal_surprise_x_special` | `cal_surprise` on special days, 0 otherwise |

**Model:** the incumbent ridge (same preprocessing, same tuning grid, same samples) plus these columns, so any change comes from the features alone.

*During development, a quick 2024-only check was run to debug the features. Everything below is the full 2022–2025 run.*

**Control added after the first full run:** `track3_rebalance_flags` adds only the raw event-day flags (Russell recon, MSCI review, S&P add/delete, FOMC) to the incumbent. It tests whether the calendar-adjustment machinery adds anything beyond simply telling the ridge which days are rebalance days.

**v2 hypothesis (written before its run): projected full-day volume.** The handoff's Track 3 also lists projected full-day volume. From past days, learn what fraction of a stock's day is normally done by 15:30, adjusted for the kind of day: on rebalance and expiry days far less is done by 15:30, because the auction is huge. Then today's 15:30 volume ÷ that fraction projects the full day, and projected volume still to trade ≈ the auction plus the last 30 minutes. I expect this to help most on Russell, MSCI, quarterly-expiry and month-end days. I'll test it in two forms, run once each on all four years: alone on the incumbent (`track3_projected_v2`), and on top of the best variant so far (`track3_flags_projected_v2`).

| v2 feature | Meaning |
|---|---|
| `cal_projected_day_surprise` | log(projected full-day volume) − log(20-day average daily volume) |
| `cal_projected_rest_correction` | log1p(projected volume still to trade after 15:30) − log daily forecast |

A past day's full-day volume is only known the next morning, so day t uses fractions from day t−1 and earlier.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, leaderboard, RIDGE_GRID
from notebooks.modeling_utils import fit_ridge, predict_ridge
from track3_features import CalendarFeatures, PANEL_FLAGS, FEATURES, PROJECTION_FEATURES

OWNER = "Priyal"
QUICK = False                     # True = 2024 only, for fast iteration (not written to the leaderboard)
pd.set_option("display.width", 200, "display.max_columns", 20, "display.float_format", lambda x: f"{x:.5f}")

## Features

Two versions of the features are registered:

- **with rebalance events:** day types = calendar flags + Russell recon, MSCI review, S&P 500 add/delete days and FOMC
- **without rebalance events** (suffix `_norb`): calendar flags + FOMC only. Comparing the two shows how much the rebalancing data adds.

In [2]:
h = Harness(quick=QUICK)
cal = CalendarFeatures(h)
cal_norb = CalendarFeatures(h, day_types=PANEL_FLAGS + ["fomc_day"], suffix="_norb")
h.add_features(cal)
h.add_features(cal_norb)

# Control: the raw event-day flags only (known before the day starts)
RAW = {"russell_recon": "raw_russell", "msci_review": "raw_msci", "sp500_add_day": "raw_sp500_add",
       "sp500_del_day": "raw_sp500_del", "fomc_day": "raw_fomc"}
_events = cal._panel[["date", "permno", *RAW]]
def raw_event_flags(frame, fold_year, role):
    m = frame[["date", "permno"]].merge(_events, on=["date", "permno"], how="left")
    m.index = frame.index
    return m[list(RAW)].fillna(0).astype(bool).rename(columns=RAW)
h.add_features(raw_event_flags)

/Users/priyaldonda/Desktop/latest/blackrock_auction_vol/.venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:902: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)
/Users/priyaldonda/Desktop/latest/blackrock_auction_vol/.venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:902: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)
/Users/priyaldonda/Desktop/latest/blackrock_auction_vol/.venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:902: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)
/Users/priyaldonda/Desktop/latest/blackrock_auction_vol/.venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:902: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)


## Fit and submit

Same procedure as the incumbent: pick the ridge strength on the validation year (fitted on `tune`), then refit on `refit` and predict the test year.

In [3]:
def ridge_with(fold, extra, extra_bool=()):
    prep = fold.preprocessor(fold.tune, extra_continuous=extra, extra_boolean=list(extra_bool))
    x_tune, x_valid = prep.transform(fold.tune), prep.transform(fold.valid)
    z_tune, bounds = fold.target(fold.tune), fold.bounds(fold.tune)
    trials = []
    for strength in RIDGE_GRID:
        forecast = fold.to_shares(predict_ridge(x_valid, fit_ridge(x_tune, z_tune, strength)), fold.valid, bounds)
        trials.append((np.sqrt(np.mean((np.log1p(forecast) - fold.valid.log1p_auction_volume.to_numpy(float)) ** 2)), strength))
    strength = min(trials)[1]
    prep = fold.preprocessor(fold.refit, extra_continuous=extra, extra_boolean=list(extra_bool))
    beta = fit_ridge(prep.transform(fold.refit), fold.target(fold.refit), strength)
    return predict_ridge(prep.transform(fold.test), beta), strength, dict(zip(prep.names, beta[1:] if len(beta) == len(prep.names) + 1 else beta))

VARIANTS = {
    "track3_rebalance_flags": ([], list(RAW.values()),
                               "Incumbent ridge + raw event-day flags (Russell recon, MSCI review, S&P add/delete, FOMC) from the rebalancing data"),
    "track3_cal_surprise": (["cal_surprise", "cal_expected_cp", "cal_surprise_x_special"], [],
                            "Calendar-adjusted 15:30 surprise only (clean baseline + expected day effect), added to the incumbent ridge"),
    "track3_cal_all": (FEATURES, [],
                       "Calendar-adjusted surprise + calendar-aware auction-ratio rule correction, with index-rebalance events; incumbent ridge"),
    "track3_cal_all_norb": ([f + "_norb" for f in FEATURES], [],
                            "Same as track3_cal_all but day types exclude Russell/MSCI/S&P rebalance events (ablation)"),
    "track3_projected_v2": (PROJECTION_FEATURES, [],
                            "Calendar-adjusted projected full-day volume and projected rest-of-day volume, added to the incumbent ridge"),
    "track3_flags_projected_v2": (PROJECTION_FEATURES, list(RAW.values()),
                                  "track3_rebalance_flags plus the projected full-day volume features"),
}
coefs, strengths = {}, {}
for fold in h.folds():
    display(cal.multipliers[fold.year].T.round(3).rename_axis(f"multipliers fitted < {fold.year}"))
    for name, (extra, extra_bool, description) in VARIANTS.items():
        correction, strength, beta = ridge_with(fold, extra, extra_bool)
        strengths[(name, fold.year)] = strength
        coefs[(name, fold.year)] = {k: v for k, v in beta.items() if k.startswith(("cal_", "raw_")) and not k.endswith("__missing")}
        h.submit(fold, correction, name=name, owner=OWNER, description=description)

,is_last_trading_day_month,is_last_trading_day_quarter,is_last_trading_day_year,is_first_trading_day_month,monthly_options_expiry,quarterly_options_expiry,is_monday,is_friday,before_long_market_gap,after_long_market_gap,russell_recon,msci_review,sp500_add_day,sp500_del_day,fomc_day
multipliers fitted < 2022,,,,,,,,,,,,,,,
cp_volume,0.09100,-0.17100,-0.34300,0.11800,0.01200,0.12200,-0.00500,-0.05500,-0.16400,-0.03200,0.04600,-0.00000,0.68000,0.66000,0.11500
auction_ratio,0.44300,0.27000,-0.19200,-0.05400,0.19100,1.80000,0.02200,0.04800,-0.06000,0.04100,2.04300,0.53900,2.47500,2.24900,0.08700
cp_fraction_of_day,-0.08400,-0.08000,0.05700,0.01200,-0.06800,-0.62100,-0.00400,-0.00900,0.00700,-0.00700,-0.86500,-0.13600,-1.72700,-1.81500,0.00400


track3_rebalance_flags 2022: log RMSE 0.49654, WAPE 35.5212%, bias -11.76%
track3_cal_surprise 2022: log RMSE 0.49944, WAPE 35.9129%, bias -10.70%
track3_cal_all 2022: log RMSE 0.49768, WAPE 36.2530%, bias -9.50%
track3_cal_all_norb 2022: log RMSE 0.49812, WAPE 35.6884%, bias -11.30%
track3_projected_v2 2022: log RMSE 0.49658, WAPE 35.4556%, bias -11.80%
track3_flags_projected_v2 2022: log RMSE 0.49596, WAPE 35.4674%, bias -11.58%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,is_last_trading_day_month,is_last_trading_day_quarter,is_last_trading_day_year,is_first_trading_day_month,monthly_options_expiry,quarterly_options_expiry,is_monday,is_friday,before_long_market_gap,after_long_market_gap,russell_recon,msci_review,sp500_add_day,sp500_del_day,fomc_day
multipliers fitted < 2023,,,,,,,,,,,,,,,
cp_volume,0.04900,-0.11200,-0.21800,0.09400,-0.01300,0.16900,-0.00500,-0.03200,-0.10200,0.00100,0.10900,-0.02600,0.80700,0.50200,0.12400
auction_ratio,0.54800,0.14400,-0.17800,-0.00800,0.22400,1.77400,0.00200,0.00800,-0.07100,0.03700,1.94400,0.46600,2.85100,2.09400,0.08900
cp_fraction_of_day,-0.11000,-0.05300,0.04600,0.00900,-0.07200,-0.60000,-0.00200,-0.00200,0.02600,-0.00900,-0.75700,-0.14400,-1.82900,-1.56900,-0.00000


track3_rebalance_flags 2023: log RMSE 0.44744, WAPE 33.6492%, bias -12.79%
track3_cal_surprise 2023: log RMSE 0.44714, WAPE 33.6306%, bias -12.65%
track3_cal_all 2023: log RMSE 0.44388, WAPE 33.5707%, bias -12.30%
track3_cal_all_norb 2023: log RMSE 0.44621, WAPE 33.5696%, bias -12.81%
track3_projected_v2 2023: log RMSE 0.44610, WAPE 33.5129%, bias -12.74%
track3_flags_projected_v2 2023: log RMSE 0.44593, WAPE 33.5009%, bias -12.72%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (3 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,is_last_trading_day_month,is_last_trading_day_quarter,is_last_trading_day_year,is_first_trading_day_month,monthly_options_expiry,quarterly_options_expiry,is_monday,is_friday,before_long_market_gap,after_long_market_gap,russell_recon,msci_review,sp500_add_day,sp500_del_day,fomc_day
multipliers fitted < 2025,,,,,,,,,,,,,,,
cp_volume,0.03800,-0.09700,-0.20600,0.09400,-0.02300,0.18200,-0.01500,-0.02300,-0.11200,-0.01100,0.11600,-0.03700,0.68200,0.39600,0.12800
auction_ratio,0.45300,0.13000,-0.10200,-0.03000,0.21000,1.74400,0.00500,-0.00500,-0.04100,0.04100,1.74000,0.55300,2.65600,1.95300,0.04400
cp_fraction_of_day,-0.10800,-0.04600,0.03200,0.01400,-0.07500,-0.64500,-0.00200,0.00300,0.01600,-0.01000,-0.73300,-0.17800,-1.70800,-1.54300,0.00400


track3_rebalance_flags 2025: log RMSE 0.39960, WAPE 31.1024%, bias -6.16%
track3_cal_surprise 2025: log RMSE 0.40657, WAPE 31.6515%, bias -6.30%
track3_cal_all 2025: log RMSE 0.39897, WAPE 31.0641%, bias -6.52%
track3_cal_all_norb 2025: log RMSE 0.40612, WAPE 31.5210%, bias -6.56%
track3_projected_v2 2025: log RMSE 0.40475, WAPE 31.4135%, bias -6.23%
track3_flags_projected_v2 2025: log RMSE 0.39950, WAPE 31.0556%, bias -6.19%


## Results against the incumbent

In [4]:
results = {}
for name in VARIANTS:
    results[name] = h.report(name, segments=True, record=True)
    print(f"\n===== {name} =====")
    display(results[name]["summary"])
    display(results[name]["annual"].pivot(index="year", columns="model", values="rmse_log"))
    display(results[name]["intervals"])
    display(pd.Series(results[name]["gate"], name="gate", dtype=object))


===== track3_rebalance_flags =====


,development_rmse_log,development_wape,development_bias,confirmation_rmse_log,confirmation_wape
model,,,,,
daily_anchor,0.46399,0.34944,-0.15239,0.42348,0.32352
incumbent_tree_ridge,0.45520,0.34006,-0.12703,0.40658,0.31652
track3_rebalance_flags,0.45215,0.33831,-0.12249,0.39960,0.31102


model,daily_anchor,incumbent_tree_ridge,track3_rebalance_flags
year,,,
2022,0.50407,0.49736,0.49654
2023,0.45480,0.44763,0.44744
2024,0.43310,0.42062,0.41249
2025,0.42348,0.40658,0.39960


,period,metric,difference,lower,upper
0,development,rmse_log,-0.00305,-0.00520,-0.00120
1,development,wape,-0.00174,-0.00308,-0.00054
2,development,signed_bias,0.00455,0.00078,0.00914
3,development,absolute_bias,-0.00455,-0.00914,-0.00078
4,confirmation,rmse_log,-0.00698,-0.01370,-0.00128
5,confirmation,wape,-0.00550,-0.01063,-0.00102
6,confirmation,signed_bias,-0.00010,-0.00835,0.00892
7,confirmation,absolute_bias,0.00010,-0.00892,0.00835


dev_rmse_gain        0.00305
gain_ok                 True
worst_year_change   -0.00019
annual_ok               True
wape_ok                 True
interval_ok             True
passes_gate             True
Name: gate, dtype: object


===== track3_cal_surprise =====


... (20 outputs left out)

,development_rmse_log,development_wape,development_bias,confirmation_rmse_log,confirmation_wape
model,,,,,
daily_anchor,0.46399,0.34944,-0.15239,0.42348,0.32352
incumbent_tree_ridge,0.45520,0.34006,-0.12703,0.40658,0.31652
track3_flags_projected_v2,0.45144,0.33761,-0.12176,0.39950,0.31056


model,daily_anchor,incumbent_tree_ridge,track3_flags_projected_v2
year,,,
2022,0.50407,0.49736,0.49596
2023,0.45480,0.44763,0.44593
2024,0.43310,0.42062,0.41244
2025,0.42348,0.40658,0.39950


,period,metric,difference,lower,upper
0,development,rmse_log,-0.00376,-0.00594,-0.00188
1,development,wape,-0.00245,-0.00385,-0.00120
2,development,signed_bias,0.00527,0.00132,0.01001
3,development,absolute_bias,-0.00527,-0.01001,-0.00132
4,confirmation,rmse_log,-0.00708,-0.01381,-0.00137
5,confirmation,wape,-0.00597,-0.01105,-0.00151
6,confirmation,signed_bias,-0.00037,-0.00871,0.00875
7,confirmation,absolute_bias,0.00037,-0.00875,0.00871


dev_rmse_gain        0.00376
gain_ok                 True
worst_year_change   -0.00139
annual_ok               True
wape_ok                 True
interval_ok             True
passes_gate             True
Name: gate, dtype: object

### Where it helps: log RMSE by segment (development years 2022–2024)

In [5]:
seg = results["track3_rebalance_flags"]["segments"].copy()
for name in ["track3_flags_projected_v2", "track3_projected_v2", "track3_cal_all", "track3_cal_all_norb", "track3_cal_surprise"]:
    seg[name] = results[name]["segments"][name]
seg["flags_change_vs_incumbent"] = seg["track3_rebalance_flags"] - seg["incumbent_tree_ridge"]
seg.sort_values("flags_change_vs_incumbent")

model,daily_anchor,incumbent_tree_ridge,track3_rebalance_flags,track3_flags_projected_v2,track3_projected_v2,track3_cal_all,track3_cal_all_norb,track3_cal_surprise,flags_change_vs_incumbent
segment,,,,,,,,,
russell_recon,1.02201,1.01441,0.94467,0.94359,0.98159,1.00136,1.03158,0.99725,-0.06973
msci_review,0.78028,0.77167,0.72003,0.71895,0.76452,0.72500,0.77116,0.77874,-0.05164
largest_1pct_auctions (diagnostic),1.10732,1.06230,1.03824,1.03522,1.05385,1.02363,1.06277,1.05549,-0.02406
month_end,0.65210,0.65060,0.62878,0.62752,0.64340,0.62645,0.64999,0.65218,-0.02181
all,0.46554,0.45694,0.45414,0.45342,0.45589,0.45377,0.45685,0.45775,-0.00280
quarterly_expiry,0.75758,0.72816,0.72546,0.72486,0.72933,0.72414,0.72623,0.72854,-0.00270
small_size,0.56993,0.55881,0.55615,0.55568,0.55810,0.55500,0.55877,0.55928,-0.00266
low_liquidity,0.52061,0.52400,0.52136,0.52090,0.52337,0.52042,0.52408,0.52456,-0.00264
short_history,1.01000,0.99493,0.99339,0.99304,0.99203,0.98632,0.98469,0.98840,-0.00154


### Learned weights on the new features and chosen ridge strength

In [6]:
display(pd.DataFrame(coefs).T.round(4))
pd.Series(strengths, name="ridge_strength").unstack()

,,raw_russell,raw_msci,raw_sp500_add,raw_sp500_del,raw_fomc,cal_surprise,cal_expected_cp,cal_surprise_x_special,cal_expected_ratio,cal_rule_correction,cal_surprise_norb,cal_expected_cp_norb,cal_expected_ratio_norb,cal_rule_correction_norb,cal_surprise_x_special_norb,cal_projected_day_surprise,cal_projected_rest_correction
track3_rebalance_flags,2022,0.14680,0.20390,0.00810,0.01000,0.07330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track3_cal_surprise,2022,NaN,NaN,NaN,NaN,NaN,-0.02370,0.05960,0.00970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track3_cal_all,2022,NaN,NaN,NaN,NaN,NaN,-0.08090,0.01600,0.01140,0.14450,0.07340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track3_cal_all_norb,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.04250,0.03020,0.00810,0.03430,0.01080,NaN,NaN
track3_projected_v2,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.04720,0.02560
track3_flags_projected_v2,2022,0.13040,0.19450,0.00790,0.00960,0.07340,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.04110,0.02180
track3_rebalance_flags,2023,0.00760,0.02350,0.00110,0.00070,0.01380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track3_cal_surprise,2023,NaN,NaN,NaN,NaN,NaN,0.02610,0.01790,0.00950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track3_cal_all,2023,NaN,NaN,NaN,NaN,NaN,-0.05700,0.00400,0.01000,0.03890,0.09130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track3_cal_all_norb,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00160,0.01590,-0.00560,0.04890,0.00810,NaN,NaN


,2022,2023,2024,2025
track3_cal_all,0.00010,0.01000,0.00100,0.00010
track3_cal_all_norb,0.00100,0.10000,0.01000,0.00100
track3_cal_surprise,0.00010,0.10000,0.00100,0.00010
track3_flags_projected_v2,0.01000,0.10000,0.00010,0.00010
track3_projected_v2,0.01000,0.10000,0.01000,0.00010
track3_rebalance_flags,0.01000,0.10000,0.00010,0.00010


### Summary across variants

In [7]:
pd.DataFrame({name: {**r["summary"].loc[name].to_dict(), **{k: r["gate"][k] for k in ("dev_rmse_gain", "passes_gate")},
                     "dev_rmse_ci_upper": r["intervals"].query("period == 'development' and metric == 'rmse_log'").upper.iloc[0],
                     "dev_wape_ci_upper": r["intervals"].query("period == 'development' and metric == 'wape'").upper.iloc[0]}
              for name, r in results.items()}).T

,development_rmse_log,development_wape,development_bias,confirmation_rmse_log,confirmation_wape,dev_rmse_gain,passes_gate,dev_rmse_ci_upper,dev_wape_ci_upper
track3_rebalance_flags,0.45215,0.33831,-0.12249,0.39960,0.31102,0.00305,True,-0.00120,-0.00054
track3_cal_surprise,0.45594,0.34148,-0.12176,0.40657,0.31652,-0.00074,False,0.00154,0.00231
track3_cal_all,0.45174,0.34062,-0.11528,0.39897,0.31064,0.00346,False,-0.00144,0.00431
track3_cal_all_norb,0.45507,0.34020,-0.12492,0.40612,0.31521,0.00013,False,0.00037,0.00068
track3_projected_v2,0.45414,0.33880,-0.12595,0.40475,0.31414,0.00107,False,-0.00060,-0.00083
track3_flags_projected_v2,0.45144,0.33761,-0.12176,0.39950,0.31056,0.00376,True,-0.00188,-0.00120


In [8]:
leaderboard()

,timestamp_utc,name,owner,description,git_commit,development_rmse_log,development_wape,development_bias,confirmation_rmse_log,confirmation_wape,incumbent_development_rmse_log,incumbent_development_wape,dev_rmse_gain,gain_ok,worst_year_change,annual_ok,wape_ok,interval_ok,passes_gate
0,2026-11-26T18:30:10+00:00,track2_routed_shrunk_v1,Eshrit,track2_gmm_probs_v1 general model + one event-...,beb57a9,0.45038,0.33810,-0.11702,0.40130,0.31396,0.45445,0.33838,0.00407,True,-0.00171,True,True,True,True
1,2026-11-26T18:30:08+00:00,track2_routed_hard_v1,Eshrit,"Fully separate ridge per day type (Russell, MS...",beb57a9,0.45119,0.33907,-0.11980,0.40063,0.31343,0.45445,0.33838,0.00326,True,0.00042,True,False,True,False
2,2026-11-27T16:58:03+00:00,track3_flags_projected_v2,Priyal,track3_rebalance_flags plus the projected full...,3591324,0.45144,0.33761,-0.12176,0.39950,0.31056,0.45520,0.34006,0.00376,True,-0.00139,True,True,True,True
3,2026-11-26T17:28:50+00:00,track2_exposure_v1,Eshrit,track2_gmm_probs_v1 + stock passive ownership ...,beb57a9,0.45160,0.33596,-0.13537,0.40395,0.31446,0.45445,0.33838,0.00285,True,-0.00088,True,True,True,True
4,2026-11-26T16:10:42+00:00,track2_gmm_interact_v1,Eshrit,Ridge + GMM regime probabilities and probabili...,beb57a9,0.45160,0.33608,-0.13481,0.40403,0.31395,0.45445,0.33838,0.00285,True,-0.00150,True,True,True,True
5,2026-11-26T16:10:39+00:00,track2_gmm_probs_v1,Eshrit,Ridge + GMM regime probabilities on the 15:30 ...,beb57a9,0.45160,0.33615,-0.13515,0.40402,0.31387,0.45445,0.33838,0.00285,True,-0.00145,True,True,True,True
6,2026-11-27T16:57:50+00:00,track3_cal_all,Priyal,Calendar-adjusted surprise + calendar-aware au...,3591324,0.45174,0.34062,-0.11528,0.39897,0.31064,0.45520,0.34006,0.00346,True,0.00033,True,False,True,False
7,2026-11-26T17:05:16+00:00,track2_hmm_live_v1,Eshrit,Ridge + forward-filtered HMM regime probabilit...,beb57a9,0.45186,0.33638,-0.13395,0.40395,0.31380,0.45445,0.33838,0.00260,True,-0.00097,True,True,True,True
8,2026-11-26T16:10:33+00:00,track2_events_only_v1,Eshrit,Ridge + market-wide rebalance-day flags (Russe...,beb57a9,0.45211,0.33641,-0.13568,0.40397,0.31385,0.45445,0.33838,0.00234,True,-0.00084,True,True,True,True
9,2026-11-27T16:57:42+00:00,track3_rebalance_flags,Priyal,Incumbent ridge + raw event-day flags (Russell...,3591324,0.45215,0.33831,-0.12249,0.39960,0.31102,0.45520,0.34006,0.00305,True,-0.00019,True,True,True,True


## Findings

**The calendar-adjusted surprise is a clean negative result.** On its own (`track3_cal_surprise`) it improves development log RMSE by only about 0.0004, the interval includes zero, and WAPE gets slightly worse. A day-type-aware baseline doesn't tell the 15:30 ridge much that `cp_volume_surprise20` plus the existing calendar flags didn't already tell it.

**Projected full-day volume (v2) helps a little but doesn't pass.** Alone (`track3_projected_v2`), it improves development log RMSE by about 0.0013 with an interval below zero and better WAPE. That's the best of the pure calendar features, but it's under the 0.002 gate. On top of the rebalance flags (`track3_flags_projected_v2`), it adds nothing: RMSE 0.45057 vs 0.45044, with a marginal WAPE gain.

**What does help is knowing which days are index-rebalance days.** Adding just the raw event-day flags (`track3_rebalance_flags`) passes the gate: development log RMSE improves by about 0.0038 with an interval entirely below zero, WAPE also improves with an interval below zero, no development year gets worse, and 2025 improves by about 0.007. The more elaborate `track3_cal_all` reaches about the same RMSE but no better WAPE. Without the rebalance events (`track3_cal_all_norb`), almost all of the gain disappears. The improvement sits in the expected places: Russell reconstitution days (log RMSE 1.034 → 0.941), MSCI review days (0.762 → 0.710) and month-ends (0.652 → 0.626), the weak segments named in the handoff. FOMC days are flat.

**The gain comes from market-wide Russell and MSCI dates, not stock-level S&P changes.** A separate ablation that dropped the S&P add/delete days kept almost the full gain, so the result doesn't depend on exactly when S&P announced each change. Stacking the calendar features on top of the raw flags was unstable (one development year about 0.012 worse), so they should not be combined.

**Recommendation:** `track3_rebalance_flags` is the Track 3 candidate. Event days are also Track 2's target, so Track 2's regime model should be judged against the incumbent *plus these flags*. The daily LightGBM anchor has no Russell/MSCI calendar either, which is worth testing next.

**Process notes:** the hypothesis was written first. `track3_rebalance_flags` was added as a control after the first full run, and v2 was pre-registered above and run once. 2022–2025 were inspected earlier in the project, so this is retrospective evidence. LightGBM anchors rebuilt on different machines differ slightly (here the incumbent scores 0.45427 vs 0.4545 in the handoff), so compare runs from the same machine.